# RQ2 --- Efficiency

In [1]:
import sys

sys.path.insert(0, "..")
import pandas as pd
from experiments._loader import load_all_results, success_only
from experiments._analysis import setup_matplotlib, SPLIT_LABEL, SPLIT_ORDER

setup_matplotlib()
df_all = load_all_results(include_baseline_fail=True)
df = success_only(df_all)
df["total_budget"] = (df["img_budget_used"] + df["txt_budget_used"]).clip(upper=df["budget_max"])
df["budget_utilization"] = df["total_budget"] / df["budget_max"]

In [2]:
METRIC_COLS = [
    "img_budget_used",
    "txt_budget_used",
    "total_budget",
    "total_evaluations",
    "runtime",
]

rows = []

for model in sorted(df_all["model"].unique()):
    for key in SPLIT_ORDER:
        mod, gmode, cat = key

        sub = df[
            (df["model"] == model)
            & (df["modality"] == mod)
            & (df["genome_mode"] == gmode)
            & (df["obj_category"] == cat)
        ]

        vals = []

        for col in METRIC_COLS:
            v = sub[col].dropna()

            if len(v):
                vals.append(f"${v.mean():.2f} \\pm {(0 if pd.isna(v.std()) else v.std()):.2f}$")
            else:
                vals.append("---")

        line = " & ".join([model, SPLIT_LABEL[key], *vals]) + r" \\"

        rows.append(line)

print("Im Budget, Txt Budget, Sum Budget, SUT Eval, Runtime")
print("\n".join(rows))

Im Budget, Txt Budget, Sum Budget, SUT Eval, Runtime
deepseek & multimodal-multi & $0.28 \pm 0.02$ & $0.18 \pm 0.12$ & $0.46 \pm 0.13$ & --- & $4.81 \pm 1.30$ \\
deepseek & multimodal-single-multi & $0.25 \pm 0.23$ & $0.10 \pm 0.09$ & $0.35 \pm 0.32$ & --- & $17.60 \pm 12.17$ \\
deepseek & multimodal-single-solo & --- & --- & --- & --- & --- \\
deepseek & image-multi & $0.27 \pm 0.01$ & $0.00 \pm 0.00$ & $0.27 \pm 0.01$ & --- & $3.57 \pm 0.34$ \\
deepseek & image-single-multi & $0.11 \pm 0.05$ & $0.00 \pm 0.00$ & $0.11 \pm 0.05$ & --- & $3.70 \pm 0.27$ \\
deepseek & image-single-solo & --- & --- & --- & --- & --- \\
deepseek & text-multi & $0.00 \pm 0.00$ & $0.57 \pm 0.37$ & $0.57 \pm 0.37$ & --- & $11.87 \pm 14.73$ \\
deepseek & text-single-multi & $0.00 \pm 0.00$ & $0.66 \pm 0.13$ & $0.66 \pm 0.13$ & --- & $1.77 \pm 0.01$ \\
deepseek & text-single-solo & --- & --- & --- & --- & --- \\
gemma & multimodal-multi & --- & --- & --- & --- & --- \\
gemma & multimodal-single-multi & --- & --